# Adversarial TTS - Class-Based Architecture

This notebook uses the refactored class-based architecture:
- **EnvironmentLoader**: Handles model loading and environment setup
- **AdversarialTrainer**: Runs the optimization loop (returns results)
- **RunLogger**: Handles all output and logging (called separately)

## 1. Imports

In [ ]:
%%bash
git clone https://github.com/Vorgesetzter/StyleTTS2
cd StyleTTS2
pip install -r requirements.txt
pip install memray
sudo apt-get install espeak-ng
git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LJSpeech
mkdir -p Audio
mv StyleTTS2-LJSpeech/Models/* Audio/
rm -rf StyleTTS2-LJSpeech

In [ ]:
%cd StyleTTS2

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch
import argparse
import memray
import numpy as np
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

from Datastructures.dataclass import ModelData
from Objectives.FitnessObjective import FitnessObjective

from Trainer.EnvironmentLoader import EnvironmentLoader
from Trainer.WaveformAdversarialTrainer import WaveformAdversarialTrainer
from Trainer.RunLogger import RunLogger

from Optimizer.pymoo_optimizer import PymooOptimizer
from pymoo.algorithms.moo.nsga2 import NSGA2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")

In [ ]:
%%bash
git pull

## 2. Configuration

Edit the values below to configure the optimization run.

In [ ]:
# =============================================================================
# Configuration — memray diagnostic run
# =============================================================================
args = argparse.Namespace(
    ground_truth_text="The birch canoe slid on the smooth planks.",
    target_text="",
    loop_count=1,
    num_generations=35,
    pop_size=100,
    batch_size=100,
    size_per_phoneme=1,
    num_rms_candidates=1,
    notify=False,
    subspace_optimization=False,
    mode="NOISE_UNTARGETED",
    objectives="PESQ=0.2, SET_OVERLAP=0.5",
)

loader = EnvironmentLoader(device)
config_data = loader.load_configuration(args)
config_data.print_summary()

tts_model, asr_model = loader.load_required_models()

audio_gt, audio_target, audio_embedding_gt, audio_embedding_target, gt_rms, target_rms = loader.generate_audio_data(
    config_data.mode, config_data.text_gt, config_data.text_target, tts_model
)

objectives = loader.initialize_objectives(
    active_objectives=config_data.active_objectives,
    model_data=ModelData(tts_model=tts_model, asr_model=asr_model),
    text_gt=config_data.text_gt,
    text_target=config_data.text_target,
    mode=config_data.mode,
    audio_gt=audio_gt,
)

trainer = WaveformAdversarialTrainer(
    tts_model, asr_model, config_data.thresholds, objectives, audio_gt, device,
    mode=config_data.mode, target_audio=audio_target,
)

print("Trainer initialized. Audio GT shape:", audio_gt.shape)

## 2. Run Optimization (with memray profiling)

In [ ]:
MEMRAY_OUTPUT = "memray_waveform.bin"

waveform_bounds = (0, 1)

optimizer = PymooOptimizer(
    bounds=waveform_bounds,
    algorithm=NSGA2,
    algo_params={"pop_size": args.pop_size},
    num_objectives=len(config_data.active_objectives),
    solution_shape=(audio_gt.shape[-1],),
)

if args.seed_target:
    n_var = audio_gt.shape[-1]
    initial_pop = np.random.uniform(waveform_bounds[0], waveform_bounds[1], (args.pop_size, n_var))
    initial_pop[0] = waveform_bounds[1]
    optimizer.update_problem((n_var,), sampling=initial_pop)

with memray.Tracker(MEMRAY_OUTPUT):
    fitness_data, archive_data, generation_count, elapsed_time_total, interrupted, generation_found = trainer.run_full_iteration(
        optimizer, args.num_generations, args.pop_size, args.batch_size
    )

print(f"[memray] Profile saved to {MEMRAY_OUTPUT}")
print(f"Generations completed: {generation_count}")

## 3. Generate Flamegraph and Download

In [ ]:
import subprocess, shutil, os

# Generate flamegraph from memray binary
result = subprocess.run(
    ["memray", "flamegraph", MEMRAY_OUTPUT, "--output", "memray_flamegraph.html", "--force"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

# Also generate stats summary
result2 = subprocess.run(
    ["memray", "stats", MEMRAY_OUTPUT, "--force"],
    capture_output=True, text=True
)
print(result2.stdout or result2.stderr)

# Download
on_kaggle = os.path.exists('/kaggle/working')
if on_kaggle:
    shutil.copy("memray_flamegraph.html", "/kaggle/working/memray_flamegraph.html")
    shutil.copy(MEMRAY_OUTPUT, f"/kaggle/working/{MEMRAY_OUTPUT}")
    print("Files copied to /kaggle/working/ — download from Output tab.")
else:
    from google.colab import files
    files.download("memray_flamegraph.html")
    files.download(MEMRAY_OUTPUT)